# Profilage DVF 2022 — Etape 2h : prix au m2 et valeurs aberrantes

Les etapes precedentes ont identifie des transactions a prix extremes (0 euro,
1 euro, 1 milliard) et documente le piege semantique de la valeur fonciere.
La mandante signale que les donnees DVF ont notoirement des problemes de fiabilite
et demande une analyse de coherence prix/surface.

Ce notebook examine :

1. Prix au m2 par type de local (mediane, P5, P95)
2. Valeurs aberrantes : prix au m2 anormalement bas (< 50 EUR/m2)
3. Valeurs aberrantes : prix au m2 anormalement eleve (> 50 000 EUR/m2)
4. Cas concrets pour verification dans Power BI

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [2]:
import duckdb
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Prix au m2 par type de local

Pour les biens batis avec un prix et une surface strictement positifs, quel est
le prix median au m2 ? Les percentiles P5 et P95 delimitent la plage normale.

In [3]:
r = con.execute(f"""
    SELECT
        \"Type local\",
        COUNT(*) AS nb,
        ROUND(APPROX_QUANTILE(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0.05), 0) AS P5,
        ROUND(APPROX_QUANTILE(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0.25), 0) AS Q1,
        ROUND(MEDIAN(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\"), 0) AS mediane,
        ROUND(APPROX_QUANTILE(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0.75), 0) AS Q3,
        ROUND(APPROX_QUANTILE(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0.95), 0) AS P95
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IS NOT NULL
    GROUP BY 1
    ORDER BY 5 DESC
""").fetchall()

col_t = 'Type de local'
col_n = 'Nb'
print(f"{col_t:<40} {col_n:>10} {'P5':>8} {'Q1':>8} {'Med.':>8} {'Q3':>8} {'P95':>8}")
print("-" * 94)
for typ, nb, p5, q1, med, q3, p95 in r:
    print(f"  {typ:<38} {nb:>8,} {p5:>6,.0f} {q1:>6,.0f} {med:>6,.0f} {q3:>6,.0f} {p95:>6,.0f}".replace(",", " "))

Type de local                                    Nb       P5       Q1     Med.       Q3      P95
----------------------------------------------------------------------------------------------
  Appartement                             636 488  1 060  2 371  3 875  7 353 115 298
  Local industriel. commercial ou assimilé  133 856    134    973  2 718  9 856 174 678
  Maison                                  754 967    603  1 453  2 234  3 429  7 402


## Cellule 4 — Valeurs aberrantes basses : prix au m2 < 50 EUR

Un prix au m2 inferieur a 50 euros pour une maison ou un appartement est
incoherent avec le marche. Ce sont probablement des cessions intrafamiliales,
des erreurs, ou des biens dans des mutations multi-lignes (piege semantique).

In [4]:
nb_bas = con.execute(f"""
    SELECT COUNT(*)
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
        AND (\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\") < 50
""").fetchone()[0]

nb_total = con.execute(f"""
    SELECT COUNT(*)
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
""").fetchone()[0]

print(f"Maisons et appartements avec prix/m2 < 50 EUR : {nb_bas:,} sur {nb_total:,} ({100.0*nb_bas/nb_total:.2f} %)".replace(",", " "))
print()

# Apercu
r = con.execute(f"""
    SELECT
        \"Date mutation\",
        \"Commune\",
        \"Code departement\",
        \"Type local\",
        \"Surface reelle bati\",
        \"Valeur fonciere\",
        ROUND(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0) AS prix_m2
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
        AND (\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\") < 50
    ORDER BY prix_m2 ASC
    LIMIT 20
""").fetchall()

col_d = 'Date'
col_c = 'Commune'
col_dp = 'Dept'
col_t = 'Type'
col_s = 'Surface'
col_v = 'Prix'
col_m = 'EUR/m2'
print(f"  {col_d:<12} {col_c:<20} {col_dp:>4} {col_t:<15} {col_s:>8} {col_v:>10} {col_m:>8}")
print(f"  {'-'*80}")
for date, com, dept, typ, sb, vf, pm in r:
    print(f"  {str(date):<12} {com:<20} {dept:>4} {typ:<15} {sb:>6} m2 {vf:>8,} EUR {pm:>6,.0f}".replace(",", " "))

Maisons et appartements avec prix/m2 < 50 EUR : 6 318 sur 1 391 455 (0.45 %)

  Date         Commune              Dept Type             Surface       Prix   EUR/m2
  --------------------------------------------------------------------------------
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         21 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement         15 m2        1 EUR      0
  2022-12-09   CAULNES                22 Appartement     

## Cellule 5 — Valeurs aberrantes hautes : prix au m2 > 50 000 EUR

Un prix au m2 superieur a 50 000 euros est exceptionnel meme a Paris.
Ce sont probablement des locaux de prestige ou des erreurs de surface.

In [5]:
nb_haut = con.execute(f"""
    SELECT COUNT(*)
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
        AND (\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\") > 50000
""").fetchone()[0]

print(f"Maisons et appartements avec prix/m2 > 50 000 EUR : {nb_haut:,} sur {nb_total:,} ({100.0*nb_haut/nb_total:.2f} %)".replace(",", " "))
print()

r = con.execute(f"""
    SELECT
        \"Date mutation\",
        \"Commune\",
        \"Code departement\",
        \"Type local\",
        \"Surface reelle bati\",
        \"Valeur fonciere\",
        ROUND(\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\", 0) AS prix_m2
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
        AND (\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\") > 50000
    ORDER BY prix_m2 DESC
    LIMIT 20
""").fetchall()

col_d = 'Date'
col_c = 'Commune'
col_dp = 'Dept'
col_t = 'Type'
col_s = 'Surface'
col_v = 'Prix'
col_m = 'EUR/m2'
print(f"  {col_d:<12} {col_c:<20} {col_dp:>4} {col_t:<15} {col_s:>8} {col_v:>12} {col_m:>10}")
print(f"  {'-'*85}")
for date, com, dept, typ, sb, vf, pm in r:
    print(f"  {str(date):<12} {com:<20} {dept:>4} {typ:<15} {sb:>6} m2 {vf:>10,} EUR {pm:>8,.0f}".replace(",", " "))

Maisons et appartements avec prix/m2 > 50 000 EUR : 46 398 sur 1 391 455 (3.33 %)

  Date         Commune              Dept Type             Surface         Prix     EUR/m2
  -------------------------------------------------------------------------------------
  2022-07-28   PARIS 08               75 Appartement         13 m2 606 210 300 EUR 46 631 562
  2022-07-28   PARIS 08               75 Appartement         22 m2 606 210 300 EUR 27 555 014
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EUR 26 248 763
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EUR 26 248 763
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EUR 26 248 763
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EUR 26 248 763
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EUR 26 248 763
  2022-12-27   VANNES                 56 Appartement         12 m2 314 985 152 EU

## Cellule 6 — Repartition des aberrants bas par departement

Les prix au m2 anormalement bas sont-ils concentres dans certains departements ?

In [6]:
r = con.execute(f"""
    SELECT
        \"Code departement\",
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0
        AND \"Surface reelle bati\" > 0
        AND \"Type local\" IN ('Maison', 'Appartement')
        AND (\"Valeur fonciere\" * 1.0 / \"Surface reelle bati\") < 50
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").fetchall()

col_d = 'Dept'
col_n = 'Nb'
col_p = '%'
print(f"{col_d:<8} {col_n:>8} {col_p:>8}")
print("-" * 26)
for dept, nb, pct in r:
    print(f"  {dept:<6} {nb:>6,} {pct:>6.1f} %".replace(",", " "))

Dept           Nb        %
--------------------------
  972       503    8.0 %
  75        454    7.2 %
  59        441    7.0 %
  65        279    4.4 %
  54        256    4.1 %
  22        198    3.1 %
  44        186    2.9 %
  88        143    2.3 %
  971       132    2.1 %
  38        123    1.9 %
  34        123    1.9 %
  92        114    1.8 %
  58        112    1.8 %
  95        109    1.7 %
  33         96    1.5 %


## Cellule 7 — Le cas Abbaretz : verification de coherence

La mandante a demande si la mutation d'Abbaretz a 10 000 euros (2 appartements
de 36 m2 + 4 dependances) est coherente. A 10 000 euros pour 2 appartements,
le prix au m2 serait d'environ 139 EUR/m2 (10 000 / 72), ce qui est tres bas
meme en milieu rural. Verification dans les donnees.

In [7]:
r = con.execute(f"""
    SELECT
        \"Date mutation\",
        \"Type local\",
        \"Surface reelle bati\",
        \"Surface terrain\",
        \"Valeur fonciere\",
        \"Nature mutation\"
    FROM '{pq}'
    WHERE \"Commune\" = 'ABBARETZ'
        AND \"Valeur fonciere\" = 10000
    ORDER BY \"Date mutation\", \"Type local\"
""").fetchall()

print(f"Mutation Abbaretz a 10 000 EUR : {len(r)} lignes")
print()
col_d = 'Date'
col_t = 'Type local'
col_s = 'Surf.batie'
col_st = 'Surf.terrain'
col_v = 'Prix'
col_n = 'Nature'
print(f"  {col_d:<12} {col_t:<20} {col_s:>10} {col_st:>12} {col_v:>10} {col_n:>10}")
print(f"  {'-'*78}")
for date, typ, sb, st, vf, nat in r:
    typ_s = str(typ) if typ else '(vide)'
    sb_s = str(sb) if sb is not None else '(null)'
    st_s = str(st) if st is not None else '(null)'
    print(f"  {str(date):<12} {typ_s:<20} {sb_s:>10} {st_s:>12} {vf:>10,} {nat:>10}".replace(",", " "))

Mutation Abbaretz a 10 000 EUR : 8 lignes

  Date         Type local           Surf.batie Surf.terrain       Prix     Nature
  ------------------------------------------------------------------------------
  2022-05-23   Appartement                  36         1261     10 000      Vente
  2022-05-23   Appartement                  36         1261     10 000      Vente
  2022-05-23   Dépendance                    0         1261     10 000      Vente
  2022-05-23   Dépendance                    0         1261     10 000      Vente
  2022-05-23   Dépendance                    0         1261     10 000      Vente
  2022-05-23   Dépendance                    0         1261     10 000      Vente
  2022-05-23   (vide)                   (null)           39     10 000      Vente
  2022-05-23   (vide)                   (null)         2561     10 000      Vente


## Cellule 8 — Synthese de l'etape 2h

In [8]:
print("Synthese — Prix au m2 et valeurs aberrantes DVF 2022")
print("=" * 55)
print()
print(f"  Biens batis (maisons + appart.) avec prix et surface > 0 : {nb_total:,}".replace(",", " "))
print(f"  Dont prix/m2 < 50 EUR  (aberrants bas)  : {nb_bas:,} ({100.0*nb_bas/nb_total:.2f} %)".replace(",", " "))
print(f"  Dont prix/m2 > 50 000 EUR (aberrants hauts) : {nb_haut:,} ({100.0*nb_haut/nb_total:.2f} %)".replace(",", " "))

Synthese — Prix au m2 et valeurs aberrantes DVF 2022

  Biens batis (maisons + appart.) avec prix et surface > 0 : 1 391 455
  Dont prix/m2 < 50 EUR  (aberrants bas)  : 6 318 (0.45 %)
  Dont prix/m2 > 50 000 EUR (aberrants hauts) : 46 398 (3.33 %)


## Cellule 9 — Fermeture

In [9]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
